# NGX D-SIB Data Validation & Splicing
**MScFE 690 Capstone — Nigerian Banking Contagion Project (Group 16011)**

This notebook ingests raw daily price CSVs for the five Nigerian D-SIBs (downloaded from
Investing.com, NGX, or similar), splices predecessor/successor tickers into one continuous
series per bank, computes log returns, and produces a per-bank **data-quality report** covering:

1. **Coverage** — first/last date, trading days per year vs. the NGX calendar
2. **Gaps** — longest run of missing calendar weekdays (suspensions? missing rows?)
3. **Zero-return runs** — stale prices / the 2008–09 price-floor episode
4. **Extreme returns** — likely unadjusted corporate actions (bonus/rights ex-dates)

**Workflow:** put your downloaded CSVs in `data/raw/` using the file names in the
`BANK_CONFIG` cell below (or edit that cell to match your file names), then run all cells.
Read the quality report **before** any GARCH estimation.

**Outputs written:** `output/prices_spliced.csv`, `output/returns_log.csv`,
`output/data_quality_report.txt`

**Supported input formats (auto-detected per file):**
- Investing.com export: `Date, Price, Open, High, Low, Vol., Change %` (newest-first rows,
  dates like `12/31/2024`, numbers with commas) — handled automatically
- Generic: any CSV with a date column named `Date`/`TradeDate` and a close column named
  `Close`/`Price`/`ClosePrice`

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## 1. Configuration

Edit the file names in `BANK_CONFIG` to match what you downloaded. For each bank, list the
`(file, label)` segments in **chronological order** — the splice uses the earlier series
first, then switches to the next series at its first available date. The return on each
switch date is dropped because it spans two corporate identities.

| Bank | Ticker history to splice |
|---|---|
| First Bank | FIRSTBANK → FBNH (holdco, late 2012) → FIRSTHOLDCO (2025 rename) |
| GTBank | GUARANTY → GTCO (holdco, mid-2021) |
| Access | ACCESS → ACCESSCORP (holdco, 2022); Diamond merger 2019 |
| Zenith | ZENITHBANK throughout (bonus issues — watch ex-dates) |
| UBA | UBA throughout |

In [ ]:
RAW_DIR = "data/raw"
OUT_DIR = "output"

SAMPLE_START = "2008-01-01"
SAMPLE_END   = "2024-12-31"

BANK_CONFIG = {
    "FIRSTBANK": [
        ("fbnh.csv",       "FBNH / FIRSTHOLDCO (continuous file, incl. pre-holdco backfill)"),
    ],
    "GTB": [
        ("gtco.csv",       "GUARANTY/GTCO (continuous file)"),
    ],
    "ACCESS": [
        ("access.csv",     "ACCESS (pre-holdco, to Mar 2022)"),
        ("accesscorp.csv", "ACCESSCORP (from Mar 2022)"),
    ],
    "ZENITH": [
        ("zenithbank.csv", "ZENITHBANK"),
    ],
    "UBA": [
        ("uba.csv",        "UBA"),
    ],
}

# Thresholds for the quality report
EXTREME_RETURN     = 0.20   # |log return| above this => flag (possible unadjusted corporate action)
ZERO_RUN_MIN       = 5      # flag runs of >= this many consecutive zero returns
EXPECTED_DAYS_LOW  = 200    # trading days/year below this => coverage warning
GAP_DAYS_FLAG      = 10     # weekday gaps of >= this many days => flag

# Known corporate-event dates to annotate (approximate — verify against NGX notices)
KNOWN_EVENTS = {
    "FIRSTBANK": ["2012-11-26 FBN Holdings relisting (share-for-share holdco)"],
    "GTB":       ["2021-07-01 GTCO holdco restructuring (approx.)"],
    "ACCESS":    ["2019-04-01 Diamond Bank merger completion (approx.)",
                  "2022-05-03 Access Holdings relisting (approx.)"],
}

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("CSV files currently in data/raw/:", [os.path.basename(p) for p in glob.glob(os.path.join(RAW_DIR, "*.csv"))])

## 2. Loading and splicing functions

In [ ]:
def _find_col(cols, candidates):
    lower = {c.lower().strip(): c for c in cols}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


def load_price_csv(path):
    """Load one raw CSV -> DataFrame indexed by date with a 'close' column."""
    df = pd.read_csv(path)
    date_col  = _find_col(df.columns, ["date", "tradedate", "trade date", "pricedate"])
    close_col = _find_col(df.columns, ["price", "close", "closeprice", "close price",
                                       "closing price", "last"])
    if date_col is None or close_col is None:
        raise ValueError(f"{os.path.basename(path)}: could not find date/close columns "
                         f"(saw: {list(df.columns)})")
    out = df[[date_col, close_col]].copy()
    out.columns = ["date", "close"]

    # Numbers like "1,234.56" -> float
    if out["close"].dtype == object:
        out["close"] = out["close"].astype(str).str.replace(",", "", regex=False).str.strip()
        out["close"] = pd.to_numeric(out["close"], errors="coerce")

    out["date"] = pd.to_datetime(out["date"], errors="coerce", dayfirst=False)
    if out["date"].isna().mean() > 0.5:   # retry day-first (e.g. 31/12/2024)
        out["date"] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)

    out = (out.dropna(subset=["date", "close"])
              .drop_duplicates(subset="date", keep="last")
              .sort_values("date")
              .set_index("date"))
    return out.loc[(out.index >= SAMPLE_START) & (out.index <= SAMPLE_END)]


def splice_bank(bank, segments):
    """Concatenate segments chronologically; later segments take precedence from
    their first date onward. Returns (prices, notes, switch_dates)."""
    notes, switch_dates, spliced = [], [], None
    for fname, label in segments:
        path = os.path.join(RAW_DIR, fname)
        if not os.path.exists(path):
            notes.append(f"  MISSING FILE: {fname} ({label}) — segment skipped")
            continue
        seg = load_price_csv(path)
        if seg.empty:
            notes.append(f"  EMPTY after parsing/sample filter: {fname} ({label})")
            continue
        notes.append(f"  loaded {fname:<28s} {label:<28s} "
                     f"{seg.index[0].date()} -> {seg.index[-1].date()}  ({len(seg)} rows)")
        if spliced is None:
            spliced = seg
        else:
            switch = seg.index[0]
            switch_dates.append(switch)
            spliced = pd.concat([spliced.loc[spliced.index < switch], seg])
            notes.append(f"    spliced at {switch.date()} (return on this date will be dropped)")
    if spliced is not None:
        spliced = spliced[~spliced.index.duplicated(keep="last")].sort_index()
        spliced.columns = [bank]
    return spliced, notes, switch_dates

## 3. Quality checks

In [ ]:
def longest_true_run(mask):
    """Length and end-position of the longest run of True in a boolean sequence."""
    best = cur = 0
    best_end = -1
    for i, v in enumerate(mask):
        cur = cur + 1 if v else 0
        if cur > best:
            best, best_end = cur, i
    return best, best_end


def quality_report(bank, prices, returns, notes, switch_dates):
    lines = [f"\n{'=' * 74}", f"BANK: {bank}", "=" * 74]
    lines += notes
    for ev in KNOWN_EVENTS.get(bank, []):
        lines.append(f"  known corporate event: {ev}")

    if prices is None or prices.empty:
        lines.append("  NO DATA — nothing to check.")
        return "\n".join(lines)

    s = prices[bank].dropna()
    r = returns[bank].dropna()

    # 1) Coverage
    lines.append(f"\n  Coverage: {s.index[0].date()} -> {s.index[-1].date()}   ({len(s)} trading days)")
    if s.index[0] > pd.Timestamp("2008-01-15"):
        lines.append(f"  ** WARNING: series starts {s.index[0].date()} — regime 1 (2008-09 GFC) "
                     f"is not fully covered. You need earlier data from NGX or another source. **")
    per_year = s.groupby(s.index.year).size()
    lines.append("  Trading days per year (NGX full year is roughly 240-250):")
    for yr, n in per_year.items():
        flag = "   <-- LOW COVERAGE" if (n < EXPECTED_DAYS_LOW and
                                         yr not in (s.index[0].year, s.index[-1].year)) else ""
        lines.append(f"    {yr}: {n:4d}{flag}")

    # 2) Gaps (calendar weekdays with no observation)
    wd = pd.date_range(s.index[0], s.index[-1], freq="B")
    missing = ~wd.isin(s.index)
    gap_len, gap_end = longest_true_run(missing.tolist())
    if gap_len >= GAP_DAYS_FLAG:
        lines.append(f"\n  Longest weekday gap: {gap_len} business days, ending {wd[gap_end].date()}"
                     f"   <-- investigate (suspension? missing rows? public holidays only?)")
    else:
        lines.append(f"\n  Longest weekday gap: {gap_len} business days "
                     f"(holidays account for small gaps; nothing alarming)")

    # 3) Zero-return runs (stale prices / price floors)
    zero = (r.abs() < 1e-12)
    zrun_len, zrun_end = longest_true_run(zero.tolist())
    n_runs, cur = 0, 0
    for v in zero:
        cur = cur + 1 if v else 0
        if cur == ZERO_RUN_MIN:
            n_runs += 1
    lines.append(f"\n  Zero-return days: {int(zero.sum())} of {len(r)} ({100 * zero.mean():.1f}%)")
    lines.append(f"  Runs of >= {ZERO_RUN_MIN} consecutive zero returns: {n_runs}")
    if zrun_len >= ZERO_RUN_MIN:
        end_date = r.index[zrun_end].date()
        lines.append(f"  Longest zero-return run: {zrun_len} days, ending {end_date}")
        if 2008 <= end_date.year <= 2010:
            lines.append("    ^ consistent with the 2008-09 NSE price-floor episode; document this "
                         "in the data section — volatility in regime 1 is mechanically understated "
                         "during pinned prices.")
    zshare = zero.groupby(zero.index.year).mean()
    worst = zshare.sort_values(ascending=False).head(3)
    lines.append("  Highest zero-return share by year: "
                 + ", ".join(f"{y}: {v * 100:.0f}%" for y, v in worst.items()))

    # 4) Extreme returns (unadjusted corporate actions?)
    big = r[r.abs() >= EXTREME_RETURN].sort_values(key=lambda x: -x.abs())
    lines.append(f"\n  |log return| >= {EXTREME_RETURN:.0%}: {len(big)} day(s)")
    for dt, val in big.head(8).items():
        lines.append(f"    {dt.date()}  {val:+.3f}   <-- check NGX notices: bonus issue, rights "
                     f"issue, or reconstruction ex-date? If so, ADJUST, don't drop.")
    if switch_dates:
        lines.append("  Splice dates (returns already set to NaN there): "
                     + ", ".join(str(d.date()) for d in switch_dates))
    return "\n".join(lines)

## 4. Run the pipeline

In [ ]:
all_prices, report_chunks = [], []

for bank, segments in BANK_CONFIG.items():
    prices, notes, switch_dates = splice_bank(bank, segments)
    if prices is None:
        report_chunks.append(quality_report(bank, None, None, notes, switch_dates))
        continue
    ret = np.log(prices / prices.shift(1))
    for d in switch_dates:                       # drop cross-identity return
        if d in ret.index:
            ret.loc[d, bank] = np.nan
    all_prices.append(prices)
    report_chunks.append(quality_report(bank, prices, ret, notes, switch_dates))

if all_prices:
    prices_wide  = pd.concat(all_prices, axis=1).sort_index()
    returns_wide = np.log(prices_wide / prices_wide.shift(1))
    prices_wide.to_csv(os.path.join(OUT_DIR, "prices_spliced.csv"))
    returns_wide.to_csv(os.path.join(OUT_DIR, "returns_log.csv"))

    # Cross-bank overlap — the DCC stage needs a common sample
    common = returns_wide.dropna(how="any")
    header = ("NGX D-SIB DATA QUALITY REPORT\n"
              f"Sample window requested: {SAMPLE_START} to {SAMPLE_END}\n"
              f"Banks with data: {list(prices_wide.columns)}\n"
              f"Common (all-bank) return observations: {len(common)}"
              + (f", spanning {common.index[0].date()} -> {common.index[-1].date()}" if len(common) else "")
              + "\nNOTE: the DCC-GARCH stage requires a common sample; if one bank starts late, "
                "the whole common sample starts late.\n")
else:
    prices_wide = returns_wide = None
    header = ("NGX D-SIB DATA QUALITY REPORT\n"
              "No data files found/parsed. Place CSVs in data/raw/ and rerun.\n")

report = header + "\n".join(report_chunks) + "\n"
with open(os.path.join(OUT_DIR, "data_quality_report.txt"), "w") as f:
    f.write(report)
print(report)

## 5. Visual sanity check

Prices on a log scale (splice points and corporate actions show up as level shifts if
something went wrong) and log returns per bank (pinned-price stretches show as flat lines;
unadjusted ex-dates show as isolated spikes).

In [ ]:
import matplotlib.pyplot as plt

if prices_wide is not None and len(prices_wide.columns):
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    np.log(prices_wide).plot(ax=axes[0], lw=0.8)
    axes[0].set_title("Log prices (spliced series)")
    axes[0].legend(loc="upper left", ncol=5, fontsize=8)
    returns_wide.plot(ax=axes[1], lw=0.4, legend=False)
    axes[1].set_title("Daily log returns")
    plt.tight_layout()
    plt.show()
else:
    print("No data loaded yet — add CSVs to data/raw/ and rerun the notebook.")

## Next steps once the report is clean

1. Verify/adjust any flagged extreme returns against NGX corporate disclosures
   (bonus and rights ex-dates) — **adjust the price series, don't delete crash days**.
2. Update the approximate dates in `KNOWN_EVENTS` with exact NGX notice dates.
3. Confirm the common sample covers regime 1 (2008–09). If not, prioritise the NGX
   student-discount data purchase for the missing early years.
4. Feed `output/returns_log.csv` into the univariate GARCH stage of the main
   capstone notebook.